In [3]:
# General CRT packing for cyclotomic polynomial Phi_t(X) mod m
# ---------------------------------------------------------------

t = 13
m = 7

R = Integers(t)
P.<X> = PolynomialRing(R)

# Cyclotomic polynomial reduced mod t
Phi = P(cyclotomic_polynomial(m))

# Factor it into irreducible pieces over F_t
factors = list(Phi.factor())
F = [fac for fac, e in factors]   # assume squarefree (e=1 each)
k = len(F)

print("Phi_t(X) =", Phi)
print("Factors:")
for i, f in enumerate(F):
    print(f"  F{i+1} =", f)

# Quotient rings for each factor (the "slots")
S = P.quotient(Phi)

# ----- choose slot values (here as a list, edit as needed) -----
slot_values = [P(R.random_element()) for i in range(k)]

print("\nSlot values:")
for i, v in enumerate(slot_values):
    print(f"  m{i+1} =", v)

# ----- CRT combination -----
M = []
N = []
terms = []

for i in range(k):
    # M_i = product of all other factors
    Mi = P(1)
    for j in range(k):
        if j != i:
            Mi *= F[j]
    M.append(Mi)

    # N_i = M_i^{-1} mod F_i  (compute via quotient ring of F_i)
    Qi = P.quotient(F[i])
    Mi_in_Qi = Qi(Mi)
    Ni_in_Qi = Mi_in_Qi.inverse()
    Ni = P(Ni_in_Qi.lift())
    N.append(Ni)

    term = (P(slot_values[i]) * Mi * Ni) % Phi
    terms.append(term)

    print(f"\nM{i+1} =", Mi)
    print(f"N{i+1} =", Ni)
    print(f"term{i+1} = m{i+1}*M{i+1}*N{i+1} mod Phi =", term)

f = sum(terms) % Phi

print("\n=== Packed plaintext polynomial ===")
print("f =", f)

# ----- sanity check: decode by reducing mod each F_i -----
print("\nDecoding check (f mod F_i should equal slot value m_i):")
for i in range(k):
    Qi = P.quotient(F[i])
    decoded = Qi(f).lift()
    print(f"  f mod F{i+1} =", decoded, "  (expected", slot_values[i], ")")

Phi_t(X) = X^6 + X^5 + X^4 + X^3 + X^2 + X + 1
Factors:
  F1 = X^2 + 3*X + 1
  F2 = X^2 + 5*X + 1
  F3 = X^2 + 6*X + 1

Slot values:
  m1 = 5
  m2 = 7
  m3 = 7

M1 = X^4 + 11*X^3 + 6*X^2 + 11*X + 1
N1 = 7*X + 10
term1 = m1*M1*N1 mod Phi = 9*X^5 + 6*X^4 + 6*X^3 + 9*X^2 + 11

M2 = X^4 + 9*X^3 + 7*X^2 + 9*X + 1
N2 = 4*X + 1
term2 = m2*M2*N2 mod Phi = 2*X^5 + 12*X^4 + 12*X^3 + 2*X^2 + 7

M3 = X^4 + 8*X^3 + 4*X^2 + 8*X + 1
N3 = 2*X + 3
term3 = m3*M3*N3 mod Phi = X^5 + 3*X^4 + 3*X^3 + X^2 + 8

=== Packed plaintext polynomial ===
f = 12*X^5 + 8*X^4 + 8*X^3 + 12*X^2

Decoding check (f mod F_i should equal slot value m_i):
  f mod F1 = 5   (expected 5 )
  f mod F2 = 7   (expected 7 )
  f mod F3 = 7   (expected 7 )
